# Import things

In [2]:
import os
from dotenv import load_dotenv
# from scaper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

In [3]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")

In [4]:
message = "Hello, GPT! This is my first ever message to you! Hi!"

messages = [{"role": "user", "content": message}]

messages

[{'role': 'user',
  'content': 'Hello, GPT! This is my first ever message to you! Hi!'}]

In [11]:
openai = OpenAI()
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
response.choices[0].message.content

"Sure! I’d love to provide a snarky summary, but could you please provide the website's content or details? I need something to work with!"

In [5]:
from bs4 import BeautifulSoup
import requests

headers = {
  "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

def fetch_website_contents(url):
  """
  Return the title and contents of the website at the given url;
  truncate to 2,000 characters as a sensible limit
  """
  response = requests.get(url, headers=headers)
  soup = BeautifulSoup(response.content, "html.parser")
  title = soup.title.string if soup.title else "No title found"
  if soup.body:
    for irrelevant in soup.body(["script", "style", "img", "input"]):
      irrelevant.decompose()
    text = soup.body.get_text(separator="\n", strip=True)
  else:
    text = ""
  return (title + "\n\n" + text)[:2_000]

In [6]:
ed = fetch_website_contents("https://edwarddonner.com")
ed

'Checking your browser...\n\n'

# Types of prompts

## System prompt
=> Tells GPT what task they are performing and what tone GPT should use

## User prompt
-> The conversation starter that GPT should reply to

In [8]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [20]:
# Define our user prompt

user_prompt_prefix = """
Here are the contents of the website:
Provide a short summary of the website.
If it includes news or announcements, include those in the summary.
"""


# Messages

The API from OpenAI expects to receive messages in a particular structure. Many of the other APIs share this structure:
```python
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]
```

In [21]:
messages = [
  {"role": "system", "content": system_prompt},
  {"role": "user", "content": user_prompt_prefix}
]

# Call the API

response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
response.choices[0].message.content

"Sure! Just provide me with the contents of the website you'd like me to analyze, and I’ll get to work on that snarky summary for you!"

In [14]:
messages = [
  {"role": "system", "content": "You are a helpful assistant that can answer questions."},
  {"role": "user", "content": "What is 2 + 2"}
]

response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
response.choices[0].message.content

'2 + 2 equals 4.'

In [22]:
def messages_for(website):
  return [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt_prefix + website}
  ]

In [23]:
messages_for(ed)

[{'role': 'system',
  'content': '\nYou are a snarky assistant that analyzes the contents of a website,\nand provides a short, snarky, humorous summary, ignoring text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n'},
 {'role': 'user',
  'content': '\nHere are the contents of the website:\nProvide a short summary of the website.\nIf it includes news or announcements, include those in the summary.\nChecking your browser...\n\n'}]

In [24]:
def summarize(url):
  website = fetch_website_contents(url)
  response = openai.chat.completions.create(
    model="gpt-4o-mini",
    messages = messages_for(website)
  )
  return response.choices[0].message.content

In [27]:
summarize("https://thestacc.com/blog/llm-friendly-content/")

'Welcome to the world of SEO wizardry, where this website promises to help you dominate online rankings like a pro (or at least pretend to). With modules for blog SEO, local SEO, and social media posts that write themselves (thanks to AI), they’ve got you covered on all fronts. \n\nNeed daily rank updates, keyword research, or instant alerts for your reviews? Look no further! They even offer tools for managing 500+ locations at once—because who wouldn’t want to juggle that many businesses? \n\nFeeling fancy? Their white-label options let you slap your brand on everything, from reports to social posts. Just remember, while you\'re busy "dominating," the only thing that might feel dominated is your sanity. \n\nSo arm yourself with their SEO guides, free tools, and enough acronyms to make your head spin. Happy ranking! '

In [28]:
def display_summary(url):
  summary = summarize(url)
  display(Markdown(summary))

In [29]:
display_summary("https://thestacc.com/blog/llm-friendly-content/")

Welcome to the world of SEO wizardry, where your dreams of online dominance can be realized…for a price, of course! This site is all about SEO and local search mastery, offering an assortment of features like rank tracking, keyword research, and even a white-label dashboard so you can pretend it's all your genius at work.

You’ll find modules to help you automate your local rankings and manage multiple locations with the finesse of a juggler at a circus. Because what's better than SEO? SEO across 500+ locations, obviously! 

They also boast blogs, free tools, and an extensive glossary that probably has more acronyms than your average tech conference. So dive into their offerings unless you enjoy stumbling through the SEO wilderness without a compass. Happy ranking!

In [30]:
display_summary("https://anthropic.com")

## Summary of Anthropic's Website

Welcome to Anthropic, where they politely remind you that AI can either save the world or doom it—no pressure! This public benefit corporation is dedicated to ensuring AI is more friendly neighborhood superhero and less menacing overlord. 

Oh, and speaking of news: they've recently lifted the export controls on Fable 5 and Mythos 5, which means if you've been itching to use Fable 5, you can start tomorrow—hooray for you! They also offer a buffet of AI-related products like Claude Code and Claude Cowork, but really, it’s all about how to make AI responsible. Because who doesn’t love a good existential crisis sprinkled with a side of safety?